# Advanced Model Optimisation Exercises

In the following exercises, you will apply advanced modeling techniques to a patient dataset, allowing you to predict disease type, severity, and treatment outcomes.

## 📌 Table of Contents

1. [Data Preparation](#1)
2. [Cross Validation](#2)
3. [Hyperparameter Tuning](#3)
4. [Pruning](#4)
5. [Missing Data Handeling](#5)
6. [Feature Engineering](#6)
7. [Feature Selection](#7)
8. [Regularisation](#8)
9. [MLflow](#9)

**Importing packages**

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd
from scipy.stats import randint, uniform
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFECV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

import xgboost
from xgboost import XGBClassifier

import optuna
from optuna.integration import OptunaSearchCV
from optuna.distributions import IntDistribution, FloatDistribution
from optuna.samplers import TPESampler, NSGAIISampler
from optuna.pruners import SuccessiveHalvingPruner

import mlflow
from badge_3203.mlflow import (
    start_mlflow,
    stop_mlflow,
    start_experiment,
    log_cross_validation_to_mlflow,
    log_search_to_mlflow,
)

from badge_3203.evaluate import evaluate_model


## 1. Data Preparation <a class="anchor" id="1"></a>

Note: you may simple run these cells and start at Chapter 2: Crossvalidation and start at exercise a). Do make sure to briefly inspect the data frame resulting from these steps so you know what you are working with. 

**Loading and Inspecting data**

In [ ]:
df = pd.read_csv("../Data/raw/disease_nan.csv")

In [ ]:
df.shape

In [ ]:
df.head(10)

**Inspecting categorical variables**

In [ ]:
for col in df.select_dtypes(include=["object", "category"]).columns:
    print("\n" + "-"*40)
    print(f"Column: {col}")
    print(f"Number of unique values: {df[col].nunique(dropna=False)}")
    print("Value counts:")
    print(df[col].value_counts(dropna=False))

**Inspecting Numerical Variables**

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
numeric_cols = numeric_cols.drop("Patient_ID", errors="ignore")

df[numeric_cols].describe()


**Disentangeling Blood Pressure**

In [ ]:
df[["BP_systolic", "BP_diastolic"]] = (
    df["Blood_Pressure_mmHg"]
    .str.split("/", expand=True)
    .astype(float)
)


In [ ]:
df = df.drop(columns="Blood_Pressure_mmHg")

**Dummifying categorical variables**

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns

dummies = pd.get_dummies(
    df[cat_cols],
    drop_first=True
)

df = pd.concat([df, dummies], axis=1)

df = df.replace({True: 1, False: 0})

In [ ]:
df.head(50)

**Creating Binary Variable to serve as clinically meaningful targets**

In [ ]:
df["disease"] = (df["Diagnosis"] != "Healthy").astype(int)


In [ ]:
df["Hospitalisation"] = (df["Treatment_Plan"] == "Hospitalization and medication").astype(int)

In [ ]:
df["high_risk_case"] = (
    df["Severity"].isin(["Severe", "Moderate"]) &
    (df["Diagnosis"] == "Pneumonia")
).astype(int)

In [ ]:
df["high_risk_case"].value_counts()


**Cleaned and prepared Data**

In [ ]:
df.head(50)

**Defining Target and Predictors**

There are different things you can predict within this dataset that might be of clinical interest. Choose which of the following interests you the most 

- Predicting whether the patient is healthy or gets diagnosed with an illness
- Predicting whether the patient has an illness of severe severity
- Predicting whether the patient will need Hospitalisation

Specify your taregt and prediction variable accordingly

In [ ]:
# Define target
y = df["Severity_Severe"]

# Keep only numerical features
X = df.select_dtypes(include="number").copy()

# Remove target and other leakage columns (e.g. that carry direct info about your taregt that you wouldn't be able to access in a realistic model employment setting) and Patient ID 
leakage_cols = [
    col for col in X.columns
    if col.startswith("Diagnosis_")
    or col.startswith("Severity")
    or col.startswith("Treatment")
    or col.startswith("hospital")
]

# Drop target, ID, and leakage-related columns
X = X.drop(columns=["disease", "Patient_ID", "high_risk_case", "Hospitalisation"] + leakage_cols, errors="ignore")

# Print features used for training 
print("Remaining feature columns:")
print(X.columns)

# 2. Cross-validation <a class="anchor" id="2"></a>

# Exercise a) implementing simple cross-validation

For the first exercise, we will optimize the hyperparameters for our xgboost model. You can find more information about xgboost and its different parameters in the [documentation](https://xgboost.readthedocs.io/en/stable/parameter.html) and some more guidance on [tuning](https://xgboost.readthedocs.io/en/stable/tutorials/param_tuning.html). 

Decide on the hyperparameters you would like to try out. We would recommend keeping the search space smaller than 30 options to prevent long run times. 

In [ ]:
# Define XGboost model
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
)

# Specify hyperparameter grid

param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

# 5 fold cross validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    verbose=1
)

# Rund Gridsearch
grid.fit(X, y)

# Results
print("Best CV score:", grid.best_score_)
print("Best parameters:", grid.best_params_)

best_model = grid.best_estimator_

## Exercise b) Implementing nested cross-validation

In this exercise, we go one step further by selecting not only the best hyperparameters but also the best model. To do this, we use nested cross-validation, where the inner loop tunes hyperparameters and the outer loop provides an unbiased estimate of each model’s performance. This separation ensures an unbiased performance estimate and allows us to reliably compare different model types. For your comparison, focus on xgboost, decision tree and random forest. 

In [ ]:

# Define models and hyperparameter grids 

models_and_grids = {
    "xgboost": (
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ),
        {
            "n_estimators": [200, 500],
            "max_depth": [3, 5],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
        }
    ),
    "random_forest": (
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        ),
        {
            "n_estimators": [200, 500],
            "max_depth": [None, 5, 10],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2],
        }
    ),
    "decision_tree": (
        DecisionTreeClassifier(
            random_state=42
        ),
        {
            "max_depth": [None, 3, 5, 10],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 5],
        }
    ),
}

# Nested CV setup:
# - Outer CV: unbiased evaluation
# - Inner CV: hyperparameter tuning for each model
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

outer_scores = []
outer_best_model_names = []
outer_best_params = []

# Run nested CV
for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    best_inner_score = -np.inf
    best_name = None
    best_estimator = None
    best_params = None

    # INNER LOOP: choose best model + hyperparameters on training fold 
    for name, (estimator, param_grid) in models_and_grids.items():
        grid = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring="roc_auc",      
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            verbose=0
        )
        grid.fit(X_train, y_train)

        # Compare best tuned version of each model using INNER CV score
        if grid.best_score_ > best_inner_score:
            best_inner_score = grid.best_score_
            best_name = name
            best_estimator = grid.best_estimator_
            best_params = grid.best_params_

    # OUTER LOOP: evaluate chosen model on held-out outer test fold 
    # GridSearchCV already refit best_estimator on all X_train, y_train
    y_proba = best_estimator.predict_proba(X_test)[:, 1]
    fold_auc = roc_auc_score(y_test, y_proba)

    outer_scores.append(fold_auc)
    outer_best_model_names.append(best_name)
    outer_best_params.append(best_params)

    print(f"\nOuter fold {fold}")
    print("  Selected model:", best_name)
    print("  Best inner-CV AUC:", round(best_inner_score, 4))
    print("  Outer test AUC:", round(fold_auc, 4))
    print("  Best params:", best_params)

# Summarize nested CV results 

print("\n" + "="*60)
print("Nested CV results (outer folds):")
print("AUC per fold:", [round(s, 4) for s in outer_scores])
print("Mean AUC:", round(np.mean(outer_scores), 4))
print("Std  AUC:", round(np.std(outer_scores), 4))

print("\nModel selection frequency:")
print(pd.Series(outer_best_model_names).value_counts())

# (Optional) Inspect which params were chosen each fold
results_df = pd.DataFrame({
    "outer_fold_auc": outer_scores,
    "selected_model": outer_best_model_names,
    "best_params": outer_best_params
})
results_df

# 3. Hyperparameter Tuning <a class="anchor" id="3"></a>

## Exercise c) Implementing Random Search for hyperparameter tuning

In the previous exercises we used grid search for hyperparameter tuning. However, we've already learned that a random search is often more efficient than a grid search, so let's implement it!


Unlike grid search, you'll define the parameter space using distributions to sample from. If you supply an array it will be sampled uniformly. You can also use other distributions, like loguniform, which is often used for parameters like learning_rate.
See [scipy distributions](https://docs.scipy.org/doc/scipy/reference/stats.html) for a full list of distributions you can use.

Define your search space using these distributions.

Unlike grid search, the parameter space for a random search does not define the exact number of trials. You need to decide how many iterations to run based on how much time you want to invest. Remember, each trial fits multiple models for cross-validation, so we recommend keeping the number of trials under 30 to keep the runtime quick enough.


In [ ]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Define random search space 

# param_distributions = {
#     "n_estimators": [100, 200, 300, 500, 800],
#     "max_depth": [2, 3, 4, 5, 6],
#     "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
#     "subsample": [0.6, 0.8, 1.0],
#     "colsample_bytree": [0.6, 0.8, 1.0],
#     "min_child_weight": [1, 3, 5, 7],
#     "gamma": [0, 0.1, 0.3, 1.0],
# }

param_distributions = {
    "n_estimators": randint(100, 1000),
    "max_depth": randint(2, 8),
    "learning_rate": uniform(0.01, 0.29),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 8),
    "gamma": uniform(0.0, 2.0),
}

# 5-fold cross validation

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# Random search with CV

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_distributions,
    n_iter=30,            # number of random configs to try (increase for more thorough search)
    scoring="roc_auc",    # change to "accuracy" if your course requires it
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X, y)


# Results 

print("Best CV score:", random_search.best_score_)
print("Best parameters:", random_search.best_params_)

best_model = random_search.best_estimator_

## Exercise d) Different optimization techniques: Optuna - Bayesian
As you now have already seen, there are different optimization techniques with different structures. We'll now be using Optuna which implements some more complex hyperparameter search algorithms. You can find more information about the different algorithms in the documentation: [optuna algorithms](https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/003_efficient_optimization_algorithms.html). The standard algorithm is a Tree-structured Parzen Estimator (TPE), which is a type of bayesian estimator.

Optuna implements a similar cross-validation hyperparameter optimization function as sklearn: [OptunaSearchCV](https://optuna.readthedocs.io/en/v2.0.0/reference/generated/optuna.integration.OptunaSearchCV.html). However, the parameter distributions are defined slightly differently, see [optuna distributions](https://optuna.readthedocs.io/en/stable/reference/distributions.html) for the available distributions.

In [ ]:
# Model 
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Optuna (Bayesian/TPE) search space 
param_distributions = {
    "n_estimators": IntDistribution(100, 1000),
    "max_depth": IntDistribution(2, 8),
    "learning_rate": FloatDistribution(0.01, 0.30, log=True),
    "subsample": FloatDistribution(0.6, 1.0),
    "colsample_bytree": FloatDistribution(0.6, 1.0),
    "min_child_weight": IntDistribution(1, 10),
    "gamma": FloatDistribution(0.0, 2.0),
}

# 5-fold CV 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Bayesian optimization via OptunaSearchCV (default sampler is TPE) 
optuna_search = OptunaSearchCV(
    estimator=xgb,
    param_distributions=param_distributions,
    n_trials=50,          # increase for a more thorough search
    scoring="roc_auc",    
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

optuna_search.fit(X, y)

print("Best CV score:", optuna_search.best_score_)
print("Best parameters:", optuna_search.best_params_)

best_model = optuna_search.best_estimator_

## Exercise e) Different optimization techniques in Optuna: Genetic Optimization

As mentioned, Optuna also supports other optimization algorithms, including a genetic algorithm. You can implement this by specifying a sampler in the optimization process

You can add any specifications for the genetic sampler here: [NSGAIISampler](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.NSGAIISampler.html). 

In [ ]:
# Model
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Optuna search space
param_distributions = {
    "n_estimators": IntDistribution(100, 1000),
    "max_depth": IntDistribution(2, 8),
    "learning_rate": FloatDistribution(0.01, 0.30, log=True),
    "subsample": FloatDistribution(0.6, 1.0),
    "colsample_bytree": FloatDistribution(0.6, 1.0),
    "min_child_weight": IntDistribution(1, 10),
    "gamma": FloatDistribution(0.0, 2.0),
}

# 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Genetic sampler (NSGA-II)
sampler = NSGAIISampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)

# OptunaSearchCV using the genetic sampler
optuna_search = OptunaSearchCV(
    estimator=xgb,
    param_distributions=param_distributions,
    n_trials=50,
    scoring="roc_auc",
    cv=cv,
    study=study,
    n_jobs=-1,
    verbose=1,
)

optuna_search.fit(X, y)

print("Best CV score:", optuna_search.best_score_)
print("Best parameters:", optuna_search.best_params_)

best_model = optuna_search.best_estimator_

# 4. Pruning <a class="anchor" id="4"></a>

## Exercise f) Pruning the optimization for efficiency 
Now it’s time to speed up our hyperparameter optimization by pruning away unpromising trials early, so the strongest configurations get more compute.
Implement Optuna with pruning using the SuccessiveHalvingPruner. Take not of how many trials were pruned versus completed. W

In [ ]:
# Define the XGBoost model
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Define the hyperparameter search space
param_distributions = {
    "n_estimators": IntDistribution(100, 1000),
    "max_depth": IntDistribution(2, 15),
    "learning_rate": FloatDistribution(0.01, 0.30, log=True),
    "subsample": FloatDistribution(0.6, 1.0),
    "colsample_bytree": FloatDistribution(0.6, 1.0),
    "min_child_weight": IntDistribution(1, 10),
    "gamma": FloatDistribution(0.0, 2.0),
}

# Define the Optuna sampler and successive halving pruner
sampler = TPESampler(seed=42)
pruner = SuccessiveHalvingPruner()

# Create an Optuna study using pruning
study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner
)

# Set up OptunaSearchCV with pruning enabled
optuna_search = OptunaSearchCV(
    estimator=xgb,
    param_distributions=param_distributions,
    n_trials=50,
    scoring="roc_auc",
    cv=cv,
    study=study,
    n_jobs=-1,
    verbose=1,
)

# Run the hyperparameter optimization
optuna_search.fit(X, y)

# Count completed and pruned trials
pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
complete = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)

# Print results
print("Best CV score:", optuna_search.best_score_)
print("Best parameters:", optuna_search.best_params_)
print("Trials complete:", complete)
print("Trials pruned:", pruned)

# Store the best trained model
best_model = optuna_search.best_estimator_

# 5. Missing Data Handeling <a class="anchor" id="5"></a>

In most cases, the data is not as complete as the one we have been wroking with in these exercises. Many times, there will be missing data. In this following exercise we will therefore work with the same dataset, however, now containing missing values in multiple columns. 

In [ ]:
df_nan = pd.read_csv("../data/raw/disease_nan.csv")

In [ ]:
# Missing values per column
missing_summary = pd.DataFrame({
    "missing_count": df_nan.isna().sum(),
    "missing_percent": df_nan.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

missing_summary


## Exercise g) identify the missingness mechanism 

Use the tools discussed in the presentation to figure out what the underlying mechnisms of the missing data may be

**Analysis of Oxigen Saturation Missingness**

In [ ]:
# Missingness indicator for blood oxygen
df_nan["O2_missing"] = df_nan["Oxygen_Saturation_%"].isna().astype(int)

# Summary stats: compare key variables when O2 is missing vs observed
vars_to_check = ["Age", "Heart_Rate_bpm", "Body_Temperature_C"]
summary = df_nan.groupby("O2_missing")[vars_to_check].agg(["count", "mean", "median", "std"])
summary

In [ ]:
# Plot: missing rate by age bins (this should show higher missingness for younger patients)

df_nan["Age"] = pd.to_numeric(df_nan["Age"], errors="coerce")
df_nan["O2_missing"] = df_nan["Oxygen_Saturation_%"].isna().astype(int)

df_plot = df_nan.dropna(subset=["Age"]).copy()

age_bins = pd.cut(
    df_plot["Age"],
    bins=[0, 18, 25, 35, 45, 55, 65, 120],
    include_lowest=True
)

missing_by_age = df_plot.groupby(age_bins)["O2_missing"].mean()

missing_by_age.index = missing_by_age.index.astype(str)

missing_by_age.plot(marker="o")
plt.xticks(rotation=45, ha="right")
plt.xlabel("Age group")
plt.ylabel("Proportion Oxygen_Saturation_% missing")
plt.title("Oxygen missingness by age group")
plt.tight_layout()
plt.show()



--> Missigness seems to depend on Age; blood oxygen for young people is often not recorded (MAR)

**Analysis of Heart rate missingness**

In [ ]:
# Create missingness indicator for heart rate
df_nan["HR_missing"] = df_nan["Heart_Rate_bpm"].isna().astype(int)

# Plot distribution of observed heart rate values
plt.figure()
df_nan.loc[df_nan["HR_missing"] == 0, "Heart_Rate_bpm"].hist(bins=30)
plt.xlabel("Heart Rate (bpm)")
plt.ylabel("Count")
plt.title("Distribution of observed Heart Rate values")
plt.tight_layout()
plt.show()


--> We can see missingness depends on the value itself; only unhealthy ranges of heart rate are noted in the health record (MNAR)

**Analysis of Age Missingness**

In [ ]:
df_nan["Age"] = pd.to_numeric(df_nan["Age"], errors="coerce")
df_nan["Age_missing"] = df_nan["Age"].isna().astype(int)

numeric_cols = df_nan.select_dtypes(include="number").columns

corr_with_missing = (
    df_nan[numeric_cols]
    .corr(numeric_only=True)["Age_missing"]
    .drop("Age_missing")
    .sort_values(ascending=False)
)

corr_with_missing

In [ ]:
# Source - https://stackoverflow.com/a
# Posted by Sadegh
# Retrieved 2026-01-09, License - CC BY-SA 4.0

from scipy.stats import chi2

def little_mcar_test(data, alpha=0.05):
    """
    Performs Little's MCAR (Missing Completely At Random) test on a dataset with missing values.
    
    Parameters:
    data (DataFrame): A pandas DataFrame with n observations and p variables, where some values are missing.
    alpha (float): The significance level for the hypothesis test (default is 0.05).
    
    Returns:
    A tuple containing:
    - A matrix of missing values that represents the pattern of missingness in the dataset.
    - A p-value representing the significance of the MCAR test.
    """
    
    # Calculate the proportion of missing values in each variable
    p_m = data.isnull().mean()
    
    # Calculate the proportion of complete cases for each variable
    p_c = data.dropna().shape[0] / data.shape[0]
    
    # Calculate the correlation matrix for all pairs of variables that have complete cases
    R_c = data.dropna().corr()
    
    # Calculate the correlation matrix for all pairs of variables using all observations
    R_all = data.corr()
    
    # Calculate the difference between the two correlation matrices
    R_diff = R_all - R_c
    
    # Calculate the variance of the R_diff matrix
    V_Rdiff = np.var(R_diff, ddof=1)
    
    # Calculate the expected value of V_Rdiff under the null hypothesis that the missing data is MCAR
    E_Rdiff = (1 - p_c) / (1 - p_m).sum()
    
    # Calculate the test statistic
    T = np.trace(R_diff) / np.sqrt(V_Rdiff * E_Rdiff)
    
    # Calculate the degrees of freedom
    df = data.shape[1] * (data.shape[1] - 1) / 2
    
    # Calculate the p-value using a chi-squared distribution with df degrees of freedom and the test statistic T
    p_value = 1 - chi2.cdf(T ** 2, df)
    
    # Create a matrix of missing values that represents the pattern of missingness in the dataset
    missingness_matrix = data.isnull().astype(int)
    
    # Return the missingness matrix and the p-value
    return missingness_matrix, p_value



In [ ]:
cols_for_mcar = [
    "Age",
    "Heart_Rate_bpm",
    "Body_Temperature_C",
    "Oxygen_Saturation_%",
]

mcar_data = df_nan[cols_for_mcar].copy()
mcar_data = mcar_data.apply(pd.to_numeric, errors="coerce")

missingness_matrix, p_values = little_mcar_test(mcar_data)

p_value = np.nanmean(p_values)

print("Little's MCAR test p-value:", p_value)

if p_value > 0.05:
    print("Fail to reject H0: missingness is consistent with MCAR.")
else:
    print("Reject H0: missingness is NOT consistent with MCAR.")

--> Little's MCAR test shows that age is likely missing completely at random

## Exercise h) decide how to handle the missing data

Based on your hypotheses of exercise g) select an appropriate method for handeling the missing data. 

**Handling MCAR (Age)**

There are multiple things we can do here, since the missingness is not structural we could for example impute the values with mean/ median. In this case, the mssingness is not that high, so we can also simply drop the rows wih the missing values 

In [ ]:
df_nan = df_nan.dropna(subset=["Age"])

**Handling MAR (Oxygen Level)**

Because oxygen saturation is Missing At Random (MAR) and the missingness depends on an observed variable (Age), the best approach is to impute using other observed information (e.g., age and vitals) and optionally add a missingness indicator so the model can learn the “measurement not taken” pattern.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


# Ensure numeric types for imputation
df_imp = df_nan.copy()
df_imp["Age"] = pd.to_numeric(df_imp["Age"], errors="coerce")
df_imp["Oxygen_Saturation_%"] = pd.to_numeric(df_imp["Oxygen_Saturation_%"], errors="coerce")
df_imp["Heart_Rate_bpm"] = pd.to_numeric(df_imp["Heart_Rate_bpm"], errors="coerce")
df_imp["Body_Temperature_C"] = pd.to_numeric(df_imp["Body_Temperature_C"], errors="coerce")

# Missingness indicator for oxygen saturation
df_imp["O2_missing"] = df_imp["Oxygen_Saturation_%"].isna().astype(int)

# Iterative imputation using age + other vitals (MAR-appropriate)
impute_cols = ["Oxygen_Saturation_%", "Age", "Heart_Rate_bpm", "Body_Temperature_C"]

imp = IterativeImputer(random_state=42, max_iter=20)
df_imp[impute_cols] = imp.fit_transform(df_imp[impute_cols])

df_imp

**Handeling MNAR (heart rate)**

Heart rate being MNAR (the “healthy middle” values are systematically missing) is trickier than MAR: you generally cannot recover the true values from observed data alone, so straightforward imputation (mean/iterative) can introduce bias. The best practical approach is to (1) add a missingness indicator so the model can learn the informative “not recorded” pattern, and (2) use a conservative, domain-informed imputation (e.g., impute to a typical healthy value such as the median of the healthy range) rather than letting a model “hallucinate” values

In [ ]:
df_hr = df_nan.copy()

df_hr["Heart_Rate_bpm"] = pd.to_numeric(df_hr["Heart_Rate_bpm"], errors="coerce")

# Missingness indicator (very important for MNAR)
df_hr["HR_missing"] = df_hr["Heart_Rate_bpm"].isna().astype(int)

# Domain-informed imputation: impute missing HR to a typical "healthy" value
healthy_low, healthy_high = 65, 95

observed_healthy = df_hr["Heart_Rate_bpm"].dropna()
observed_healthy = observed_healthy[(observed_healthy >= healthy_low) & (observed_healthy <= healthy_high)]

# Fallback if the healthy range is empty in observed data
impute_value = observed_healthy.median() if len(observed_healthy) > 0 else 80.0

df_hr["Heart_Rate_bpm"] = df_hr["Heart_Rate_bpm"].fillna(impute_value)

df_hr[["Heart_Rate_bpm", "HR_missing"]].head()

# 6. Feature engineering <a class="anchor" id="6"></a>

## Exercise i) Feature Engineering

We've discussed the importance of feature engineering. Now, think about any new features you'd like to introduce to enhance your model. Consider transformations, interactions between existing features, or new derived variables that could improve the model's predictive power.

Be creative! 

After you finished your feature engineering save the dataframe as a csv to Data/processed with an informative name! This way you can retrieve your augmented dataset easily for future experimentation. 

**High Fever Indicator**

Motivation: high fever as compared to moderate fever might have implications for diagnosis or treatment 

In [ ]:
df["high_fever"] = (df["Body_Temperature_C"] >= 39.0).astype(int)

**Pulse Pressure**

Motivation: Pulse pressure is clinically meaningful and not redundant with BP alone.

In [ ]:
df["pulse_pressure"] = df["BP_systolic"] - df["BP_diastolic"]

**Age risk group (discretization)**

Motivation: Risk is non-linear in age. Buckets often work better than raw age.

In [ ]:
df["age_group"] = pd.cut(
    df["Age"],
    bins=[0, 18, 40, 65, 120],
    labels=["child", "young_adult", "adult", "elderly"]
)


**Physiological stress index (scaled composite)**

Motivation: Combines vitals on comparable scales.

In [ ]:
df["stress_index"] = (
    (df["Heart_Rate_bpm"] / 100) +
    (df["Body_Temperature_C"] / 40) +
    (1 - df["Oxygen_Saturation_%"] / 100)
)


**Tachycardia indicator (physiological stress)**

Motivation: External clinical knowledge provides established thresholds for abnormal heart rate

In [ ]:
df["tachycardia"] = (df["Heart_Rate_bpm"] > 100).astype(int)


**Saving Dataframe to processed folder**

In [ ]:
df.to_csv("../Data/processed/disease_processed.csv", index=False)


# 7. Feature Selection <a class="anchor" id="7"></a>

For the following two exercises, we'll only focus on feature selection. Use the best hyperparameters you found in the previous exercises with your xgboost model. We start applying the methods using a classic train-test split. 

**Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
)

**Defining our model**

In [ ]:
# XGBoost model with your best hyperparameters
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    colsample_bytree=1.0,
    learning_rate=0.05,
    max_depth=5,
    n_estimators=200,
    subsample=1.0,
)

## Exercise j) Filter methods 

There are many possible [Sklearn filter methods](https://scikit-learn.org/stable/modules/feature_selection.html#univariate-feature-selection) to select features. For this exercise, let's use the [f_classif](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_classif.html) function to score features based on their correlation with the target variable. We'll then select the top 6 features with the highest scores to include in the model, ensuring that we focus on the most relevant predictors

In [ ]:
# Feature selection method, use a Filter method.
feature_selection_filter = SelectKBest(score_func=f_classif, k=6)

pipeline_filter = Pipeline(
    steps=[
        ("select_features", feature_selection_filter),
        ("model", model),
    ]
)

pipeline_filter

In [ ]:
# Fit the pipeline
pipeline_filter.fit(X_train, y_train)

evaluate_model(pipeline_filter, X_train, y_train, X_test, y_test)

# # Get mask of selected features
# selected_mask = pipeline_filter.named_steps["select_features"].get_support()

# # Get feature names
# selected_features = X.columns[selected_mask]

# print("Selected features:")
# for f in selected_features:
#     print(f)

We can assess how stable our feature selection is using cross-validation. This will help us study how well the selected features perform across different subsets of the data.

In [ ]:
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results_filter = cross_validate(
    estimator=pipeline_filter,
    X=X_train,
    y=y_train,
    cv=outer_cv,
    return_estimator=True,
    return_train_score=True,
)


In [ ]:
print("CV results per fold")
display(pd.DataFrame(cv_results_filter).drop(columns=["estimator"]))

# Count how often each feature is selected
print("How many times is each feature selected")
selected_features = pd.Series(
    [
        feat
        for estimator in cv_results_filter["estimator"]
        for feat in estimator["select_features"].get_feature_names_out()
    ]
)
pd.DataFrame(selected_features.value_counts())

**Questions**
1. What are the main weaknesses of a filter method?
2. Do you notice anything about the selected features? (what could be going wrong?)
3. How many features would you like to select?

**Bonus**
1. Can you solve the problem you found in question 2?
2. How would you add hyperparameter optimization to this pipeline?

## Exercise k) Wrapper methods 

Now, let's implement Recursive Feature Elimination (RFE) with cross-validation as our wrapper method. This method recursively removes the least important features and evaluates model performance using cross-validation at each step. It allows us to identify the optimal subset of features that maximizes model performance.

We can use [RFECV()](https://scikit-learn.org/1.5/modules/generated/sklearn.feature_selection.RFECV.html) to implement this. Like SelectKBest, RFECV returns a reduced feature set, which is passed to the next step in the pipeline. Therefore, you need to place the model behind the RFECV in the pipeline, but also pass the model as a parameter to RFECV for it to use in the feature selection process.

In [ ]:
# Wrapper method using RFECV
feature_selection_wrapper = RFECV(
    estimator=model,
)

feature_selection_wrapper

# Pipeline: RFECV selects features, then the model fits on the reduced feature set
pipeline_wrapper = Pipeline(
    [
        ("select_features", feature_selection_wrapper),
        ("model", model),
    ]
)


In [ ]:
pipeline_wrapper.fit(X_train, y_train)

evaluate_model(pipeline_wrapper, X_train, y_train, X_test, y_test)

We can assess how stable our feature selection is using cross-validation. This will help us study how well the selected features perform across different subsets of the data.

In [ ]:
# It might take a little bit to run these due to the large number of features
# You can speed up the process by only using a subset of the features when loading the data at the top of this notebook
outer_cv = KFold(n_splits=3, shuffle=True, random_state=42)

cv_results_wrapper = cross_validate(
    estimator=pipeline_wrapper,
    X=X_train,
    y=y_train,
    cv=outer_cv,
    return_estimator=True,
    return_train_score=True,
)

In [ ]:
print("CV results per fold")
display(pd.DataFrame(cv_results_wrapper).drop(columns=["estimator"]))

# Count how often each feature is selected
print("How many times is each feature selected")
selected_features = pd.Series(
    [
        feat
        for estimator in cv_results_wrapper["estimator"]
        for feat in estimator["select_features"].get_feature_names_out()
    ]
)
pd.DataFrame(selected_features.value_counts())

**Questions**
1. Why would we prefer this method over sequential feature selection

# 8. Regularisation <a class="anchor" id="8"></a>

## Exercise l) Regularisation

To make a comparison, implement one xgboost model without regularisation, one with L1 regularisaton, one with L2 regularisation and one with early stopping. 

What do you obsere in terms of final performance, train-validation difference? 
Also, plot feature importance for the different models and report what you observe there. Are the results as expected? 

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# 1) Train/validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def fit_with_curves(model, X_tr, y_tr, X_val, y_val):
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_val, y_val)],
        verbose=False
    )
    return model.evals_result()

# Report final train and validation performance
def report_final_performance(evals_result, name):
    metric = list(evals_result["validation_0"].keys())[0]
    train_final = evals_result["validation_0"][metric][-1]
    val_final = evals_result["validation_1"][metric][-1]
    print(f"{name} | final train {metric}: {train_final:.4f}, final val {metric}: {val_final:.4f}")


def plot_feature_importance(model, X, title, top_n=28):
    importances = pd.Series(
        model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)

    plt.figure()
    importances.head(top_n).plot(kind="bar")
    plt.title(title)
    plt.ylabel("Feature importance")
    plt.tight_layout()
    plt.show()

# 2) Models
base_params = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    learning_rate=0.05,
    max_depth=5,
    subsample=1.0,
    colsample_bytree=1.0,
    n_estimators=1000
)

model_none = XGBClassifier(**base_params, reg_alpha=0.0, reg_lambda=0.0)
model_l1 = XGBClassifier(**base_params, reg_alpha=10.0, reg_lambda=0.0)
model_l2 = XGBClassifier(**base_params, reg_alpha=0.0, reg_lambda=10.0)
model_es = XGBClassifier(**base_params, early_stopping_rounds=5)

# 3) Fit models
evals_none = fit_with_curves(model_none, X_tr, y_tr, X_val, y_val)
evals_l1 = fit_with_curves(model_l1, X_tr, y_tr, X_val, y_val)
evals_l2 = fit_with_curves(model_l2, X_tr, y_tr, X_val, y_val)
evals_es = fit_with_curves(model_es, X_tr, y_tr, X_val, y_val)

# 4) Report final performance
report_final_performance(evals_none, "No regularisation")
report_final_performance(evals_l1, "L1 regularisation")
report_final_performance(evals_l2, "L2 regularisation")
report_final_performance(evals_es, "Early stopping")

# 5) Plot helper
def plot_curves(evals_result, title):
    metric = list(evals_result["validation_0"].keys())[0]
    train_curve = evals_result["validation_0"][metric]
    val_curve = evals_result["validation_1"][metric]

    plt.figure()
    plt.plot(train_curve, label="train")
    plt.plot(val_curve, label="validation")
    plt.xlabel("Boosting round")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.show()

# 6) Plot curves
plot_curves(evals_none, "No regularisation (baseline)")
plot_curves(evals_l1, "L1 regularisation")
plot_curves(evals_l2, "L2 regularisation")
plot_curves(evals_es, "Early stopping")

plot_feature_importance(model_none, X_tr, "Feature importance: No regularisation")
plot_feature_importance(model_l1, X_tr, "Feature importance: L1 regularisation")
plot_feature_importance(model_l2, X_tr, "Feature importance: L2 regularisation")
plot_feature_importance(model_es, X_tr, "Feature importance: Early stopping")



# 9. ML Flow <a class="anchor" id="9"></a>

## Exercise m) Structuring experiments using ML flow 

In this exercise, you will use MLflow to log the results of a simple 5-fold cross-validation experiment.
Define your target and predictor variables and set up a model using the hyperparameters you previously identified. Evaluate the model’s performance using 5-fold cross-validation, and log fold-level performance as well as the mean and standard deviation of the evaluation metric using MLflow.

In [ ]:
START_SERVER = True

process = None
if START_SERVER:
    process = start_mlflow()

# Load Processed Data
df = pd.read_csv("../Data/processed/disease_processed.csv")


# Define target and features

y = df["high_risk_case"].astype(int)

# numeric only
X = df.select_dtypes(include=[np.number]).copy()

# drop leakage columns/prefixes + target itself
drop_prefixes = ("Diagnosis", "Treatment", "Severity")
cols_to_drop = [c for c in X.columns if c.startswith(drop_prefixes)]
cols_to_drop += ["high_risk_case"]

X = X.drop(columns=cols_to_drop, errors="ignore")


# Set up MLflow experiment 
start_experiment("Exercise_CV_Basics")


# Cross validation + logging
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

with mlflow.start_run(run_name="xgb_cv_baseline"):

    # log a couple of simple params
    mlflow.log_param("model", "XGBClassifier")
    mlflow.log_param("n_splits", 5)

    fold_acc = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # fresh model each fold
        model = XGBClassifier(random_state=42, eval_metric="logloss")
        model.fit(X_train, y_train)

        preds = model.predict(X_val)
        acc = accuracy_score(y_val, preds)

        fold_acc.append(acc)
        mlflow.log_metric("accuracy_fold", acc, step=fold)

    mlflow.log_metric("accuracy_mean", float(np.mean(fold_acc)))
    mlflow.log_metric("accuracy_std", float(np.std(fold_acc)))


print("Done — check MLflow for fold + mean accuracy.")


# stop server
if process is not None:
    stop_mlflow(process)
